In [ ]:
import random

def calculate_cost(position, demand, holding_cost,
                   ordering_cost, transport_cost, capacity):

    total_cost = 0
    warehouses = len(capacity)
    products = len(demand)

    # Calculate inventory and transportation costs
    for p in range(products):
        stock = sum(position[p * warehouses:(p + 1) * warehouses])

        total_cost += stock * holding_cost[p]

        if stock > 0:
            total_cost += ordering_cost[p]

        shortage = max(0, demand[p] - stock)
        total_cost += shortage * 1000

        for w in range(warehouses):
            total_cost += (
                position[p * warehouses + w]
                * transport_cost[p][w]
            )

    # Penalize warehouse capacity violations
    for w in range(warehouses):
        stock = sum(position[p * warehouses + w]
                    for p in range(products))
        total_cost += max(0, stock - capacity[w]) * 1000

    return total_cost


def repair(position, demand, capacity):
    warehouses = len(capacity)
    products = len(demand)

    position = [max(0, round(x)) for x in position]

    # Ensure warehouse capacity is not exceeded
    for w in range(warehouses):
        while sum(position[p * warehouses + w]
                  for p in range(products)) > capacity[w]:
            indexes = [
                p * warehouses + w
                for p in range(products)
                if position[p * warehouses + w] > 0
            ]
            if not indexes:
                break
            position[indexes[0]] -= 1

    return position


def pso_inventory(demand, holding_cost, ordering_cost,
                  transport_cost, capacity, N=30, MaxIter=100):

    products = len(demand)
    warehouses = len(capacity)
    dimensions = products * warehouses

    w, c1, c2 = 0.7, 1.5, 1.5

    positions = [
        repair([random.randint(0, max(demand))
                for _ in range(dimensions)], demand, capacity)
        for _ in range(N)
    ]

    velocities = [
        [random.uniform(-2, 2) for _ in range(dimensions)]
        for _ in range(N)
    ]

    costs = [
        calculate_cost(x, demand, holding_cost,
                       ordering_cost, transport_cost, capacity)
        for x in positions
    ]

    pBest = [x[:] for x in positions]
    pBestCost = costs[:]

    best = costs.index(min(costs))
    gBest = pBest[best][:]
    gBestCost = pBestCost[best]

    for iteration in range(MaxIter):
        for i in range(N):
            for d in range(dimensions):
                r1, r2 = random.random(), random.random()

                velocities[i][d] = (
                    w * velocities[i][d]
                    + c1 * r1 * (pBest[i][d] - positions[i][d])
                    + c2 * r2 * (gBest[d] - positions[i][d])
                )

                positions[i][d] += velocities[i][d]

            positions[i] = repair(positions[i], demand, capacity)

            cost = calculate_cost(
                positions[i], demand, holding_cost,
                ordering_cost, transport_cost, capacity
            )

            if cost < pBestCost[i]:
                pBest[i] = positions[i][:]
                pBestCost[i] = cost

            if pBestCost[i] < gBestCost:
                gBest = pBest[i][:]
                gBestCost = pBestCost[i]

    return gBest, gBestCost


# Main program
if __name__ == "__main__":

    products = ["Rice", "Wheat"]
    warehouses = ["Warehouse A", "Warehouse B"]

    demand = [100, 80]
    holding_cost = [2, 3]
    ordering_cost = [100, 120]
    transport_cost = [[1, 3], [2, 1]]
    capacity = [120, 100]

    best, minimum_cost = pso_inventory(
        demand, holding_cost, ordering_cost,
        transport_cost, capacity
    )

    print("\nPSO INVENTORY AND WAREHOUSE OPTIMIZATION")
    print("----------------------------------------")

    for p, product in enumerate(products):
        print(f"\n{product}:")
        for w, warehouse in enumerate(warehouses):
            quantity = best[p * len(warehouses) + w]
            print(f" {warehouse}: {quantity} units")

    print("\nWarehouse Utilization:")
    for w, warehouse in enumerate(warehouses):
        used = sum(
            best[p * len(warehouses) + w]
            for p in range(len(products))
        )
        print(f"{warehouse}: {used}/{capacity[w]} units")

    print(f"\nMinimum Total Cost: {minimum_cost:.2f}")


PSO INVENTORY AND WAREHOUSE OPTIMIZATION
----------------------------------------

Rice:
 Warehouse A: 100 units
 Warehouse B: 0 units

Wheat:
 Warehouse A: 0 units
 Warehouse B: 80 units

Warehouse Utilization:
Warehouse A: 100/120 units
Warehouse B: 80/100 units

Minimum Total Cost: 840.00
